# LeetCode #50 — Pow(x, n): lũy thừa nhanh (Binary Exponentiation)

**Mức độ:** Medium · **Chủ đề:** Math, Recursion

Skeleton trên LeetCode:

```python
class Solution:
    def myPow(self, x: float, n: int) -> float:
```

Mục tiêu của notebook là học **lũy thừa nhanh**: tính `x^n` chỉ với khoảng `log₂ n` phép nhân thay vì `n` phép — một trong những kỹ thuật "chia để trị" kinh điển nhất, dùng trong mật mã, tính Fibonacci nhanh, nhân ma trận, …

---

## Sau notebook này, bạn sẽ nắm được

1. Vì sao nhân lặp `n` lần không chạy kịp với `n ≈ 2^31`.
2. Ý tưởng **bình phương**: `x^n = (x^(n/2))²`.
3. Cách viết **đệ quy** và **vòng lặp** (dựa trên các bit của `n`).
4. Xử lý số mũ **âm** và số mũ `0`.
5. Vì sao so sánh số thực phải dùng **sai số cho phép**.

# 1. Bài toán là gì?

Cài đặt `pow(x, n)`, tức tính `x^n`.

**Constraints (theo đề):**

- `-100.0 < x < 100.0`
- `-2^31 <= n <= 2^31 - 1`
- `n` là số nguyên.
- Hoặc `x != 0`, hoặc `n > 0`.
- `-10^4 <= x^n <= 10^4`

---

## Ví dụ

```text
x = 2.00000, n = 10   ->  1024.00000
x = 2.10000, n = 3    ->  9.26100
x = 2.00000, n = -2   ->  0.25000      (2^-2 = 1/4)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Đệ quy (recursion)
2. Chia để trị (divide and conquer)
3. Biểu diễn nhị phân và các phép toán bit

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Đệ quy (recursion)

### Đệ quy là gì?

Một hàm **đệ quy** là hàm tự gọi lại chính nó trên một bài toán **nhỏ hơn** cùng dạng. Mỗi hàm đệ quy có hai phần:

1. **Trường hợp cơ sở (base case)**: bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: chia nhỏ, gọi lại, rồi dùng kết quả để trả lời bài lớn.

Ví dụ `n! = n × (n−1)!`, với `0! = 1`.

In [1]:
def giai_thua(n):
    if n == 0:                  # trường hợp cơ sở
        return 1
    return n * giai_thua(n - 1) # bước đệ quy: bài nhỏ hơn
print([giai_thua(n) for n in range(7)])

[1, 1, 2, 6, 24, 120, 720]


### Ngăn xếp lời gọi (call stack)

Mỗi lần gọi hàm, máy tạo một **khung** (frame) chứa biến cục bộ và đặt lên **ngăn xếp lời gọi**. Hàm con chạy xong thì khung của nó bị bỏ, kết quả trả về cho khung bên dưới. Ta in thụt lề theo độ sâu để thấy rõ:

In [2]:
def giai_thua_vet(n, sau=0):
    le = "    " * sau
    print(f"{le}gọi giai_thua({n})")
    if n == 0:
        print(f"{le}→ trả 1 (cơ sở)")
        return 1
    kq = n * giai_thua_vet(n - 1, sau + 1)
    print(f"{le}→ trả {n} × ... = {kq}")
    return kq
_ = giai_thua_vet(3)

gọi giai_thua(3)
    gọi giai_thua(2)
        gọi giai_thua(1)
            gọi giai_thua(0)
            → trả 1 (cơ sở)
        → trả 1 × ... = 1
    → trả 2 × ... = 2
→ trả 3 × ... = 6


### Cách viết một hàm đệ quy đúng

1. **Định nghĩa rõ** hàm trả về gì cho đầu vào nào ("`f(node)` = chiều cao cây gốc `node`").
2. **Tin** rằng lời gọi trên bài nhỏ hơn đã trả đúng (giả thiết quy nạp).
3. Ghép kết quả các bài nhỏ thành kết quả bài lớn.
4. Kiểm tra trường hợp cơ sở được chạm tới với **mọi** đầu vào (mỗi lần gọi phải nhỏ đi).

Ví dụ: tổng các chữ số, lũy thừa nhanh.

In [3]:
def tong_chu_so(n):
    if n < 10:
        return n
    return n % 10 + tong_chu_so(n // 10)

def luy_thua(x, k):              # x^k, chia đôi số mũ
    if k == 0:
        return 1
    nua = luy_thua(x, k // 2)
    return nua * nua * (x if k % 2 else 1)

print("tong_chu_so(98765) =", tong_chu_so(98765), "| luy_thua(3, 13) =", luy_thua(3, 13), "=", 3 ** 13)

tong_chu_so(98765) = 35 | luy_thua(3, 13) = 1594323 = 1594323


### Ghi nhớ kết quả (memoization)

Một số hàm đệ quy gọi **lặp lại** cùng một bài con rất nhiều lần, ví dụ Fibonacci `F(n) = F(n−1) + F(n−2)`: `F(5)` gọi `F(3)` hai lần, `F(2)` ba lần… → `O(2^n)`.

**Memoization**: lưu kết quả đã tính vào dict (hoặc dùng `@lru_cache`), lần sau gặp lại thì trả ngay → mỗi bài con tính đúng một lần.

In [4]:
from functools import lru_cache

dem_goi = 0
def fib_cham(n):
    global dem_goi
    dem_goi += 1
    return n if n < 2 else fib_cham(n - 1) + fib_cham(n - 2)

memo = {}
def fib_memo(n):
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

@lru_cache(maxsize=None)
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)

print("fib_cham(25) =", fib_cham(25), "với", dem_goi, "lần gọi")
print("fib_memo(25) =", fib_memo(25), "| bảng nhớ có", len(memo), "mục")
print("fib_cache(90) =", fib_cache(90))

fib_cham(25) = 75025 với 242785 lần gọi
fib_memo(25) = 75025 | bảng nhớ có 24 mục
fib_cache(90) = 2880067194370816120


### Lỗi hay gặp

- **Thiếu / sai trường hợp cơ sở** → đệ quy vô hạn → `RecursionError` (Python mặc định giới hạn ~1000 tầng).
- Bài con **không nhỏ đi** (ví dụ gọi `f(n)` trong `f(n)`).
- Đệ quy quá sâu với dữ liệu lớn (danh sách 10^5 nút) → đổi sang vòng lặp hoặc tăng `sys.setrecursionlimit` (có rủi ro).
- Dùng list/biến dùng chung giữa các lời gọi mà quên "trả lại" trạng thái (xem backtracking).

In [5]:
def thieu_co_so(n):
    return thieu_co_so(n - 1)
try:
    thieu_co_so(5)
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")

RecursionError: maximum recursion depth exceeded ...


## 2.2 Chia để trị (divide and conquer)

### Ý tưởng

1. **Chia** bài toán thành các bài con **cùng dạng**, nhỏ hơn (thường là hai nửa).
2. **Trị**: giải các bài con bằng đệ quy (bài đủ nhỏ thì giải trực tiếp).
3. **Kết hợp** kết quả các bài con thành kết quả bài lớn.

Khác DP: các bài con **không chồng lấp** (mỗi phần dữ liệu thuộc đúng một bài con).

```text
            [8 phần tử]                      tầng 0: 1 bài, tổng n việc
         [4]          [4]                    tầng 1: 2 bài, tổng n việc
       [2]  [2]     [2]  [2]                 tầng 2: 4 bài, tổng n việc
      [1][1][1][1] [1][1][1][1]              log₂ n tầng  →  O(n log n)
```

In [6]:
def max_chia_de_tri(a, lo, hi, sau=0):
    """Phần tử lớn nhất của a[lo..hi]."""
    if lo == hi:
        return a[lo]                                   # bài đủ nhỏ
    mid = (lo + hi) // 2
    trai = max_chia_de_tri(a, lo, mid, sau + 1)        # trị nửa trái
    phai = max_chia_de_tri(a, mid + 1, hi, sau + 1)    # trị nửa phải
    kq = max(trai, phai)                               # kết hợp
    print("    " * sau + f"a[{lo}..{hi}] = {a[lo:hi + 1]} → max {kq}")
    return kq

_ = max_chia_de_tri([3, 8, 1, 9, 4, 7, 2, 6], 0, 7)

        a[0..1] = [3, 8] → max 8
        a[2..3] = [1, 9] → max 9
    a[0..3] = [3, 8, 1, 9] → max 9
        a[4..5] = [4, 7] → max 7
        a[6..7] = [2, 6] → max 6
    a[4..7] = [4, 7, 2, 6] → max 7
a[0..7] = [3, 8, 1, 9, 4, 7, 2, 6] → max 9


### Các ví dụ quen thuộc

| Bài | Chia | Kết hợp | Độ phức tạp |
|---|---|---|---|
| Tìm nhị phân | bỏ một nửa | không cần | `O(log n)` |
| Merge sort | hai nửa | trộn `O(n)` | `O(n log n)` |
| Lũy thừa nhanh `x^n` | `x^(n/2)` | bình phương | `O(log n)` |
| Dựng cây cân bằng từ mảng sắp xếp | phần tử giữa làm gốc | gắn hai cây con | `O(n)` |
| Dựng cây từ hai cách duyệt | tách theo vị trí gốc | gắn hai cây con | `O(n)` với hash map |

In [7]:
def luy_thua_nhanh(x, n, sau=0):
    if n == 0:
        return 1
    nua = luy_thua_nhanh(x, n // 2, sau + 1)
    kq = nua * nua * (x if n % 2 else 1)
    print("    " * sau + f"{x}^{n} = ({x}^{n // 2})²{' · ' + str(x) if n % 2 else ''} = {kq}")
    return kq

_ = luy_thua_nhanh(2, 10)

            2^1 = (2^0)² · 2 = 2
        2^2 = (2^1)² = 4
    2^5 = (2^2)² · 2 = 32
2^10 = (2^5)² = 1024


Lỗi hay gặp: chia không đều hoặc không nhỏ đi (`mid` tính sai → đệ quy vô hạn); tạo bản sao `a[:mid]` ở mỗi tầng (tốn thêm `O(n log n)` bộ nhớ — truyền chỉ số `lo, hi` thay vì cắt list); quên trường hợp đoạn rỗng.

## 2.3 Biểu diễn nhị phân và các phép toán bit

### Số nguyên dưới dạng bit

Máy tính lưu số nguyên ở hệ **nhị phân**: mỗi chữ số (bit) là 0 hoặc 1, bit thứ `k` (đếm từ 0, bên phải) có giá trị `2^k`.

```text
13 = 8 + 4 + 1 = 1·2³ + 1·2² + 0·2¹ + 1·2⁰ = 0b1101
bit:              3     2      1      0
```

In [8]:
x_bit = 13
print("bin(13) =", bin(x_bit), "| format 8 bit:", format(x_bit, "08b"), "| int('1101', 2) =", int("1101", 2))
print("bit thứ k của 13:", [(x_bit >> k) & 1 for k in range(4)], "(từ bit 0)")

bin(13) = 0b1101 | format 8 bit: 00001101 | int('1101', 2) = 13
bit thứ k của 13: [1, 0, 1, 1] (từ bit 0)


### Sáu phép toán bit

| Phép | Ký hiệu | Quy tắc từng bit | Ví dụ `12 = 1100`, `10 = 1010` |
|---|---|---|---|
| AND | `a & b` | 1 nếu **cả hai** là 1 | `1000` = 8 |
| OR | `a \| b` | 1 nếu **ít nhất một** là 1 | `1110` = 14 |
| XOR | `a ^ b` | 1 nếu **khác nhau** | `0110` = 6 |
| NOT | `~a` | đảo mọi bit | `~12 = -13` (Python) |
| dịch trái | `a << k` | thêm `k` số 0 bên phải = nhân `2^k` | `12 << 1 = 24` |
| dịch phải | `a >> k` | bỏ `k` bit phải = chia nguyên `2^k` | `12 >> 2 = 3` |

In [9]:
a_b, b_b = 12, 10
for ten, v in [("a & b", a_b & b_b), ("a | b", a_b | b_b), ("a ^ b", a_b ^ b_b), ("a << 1", a_b << 1), ("a >> 2", a_b >> 2), ("~a", ~a_b)]:
    print(f"{ten:<7} = {v:>4}   bin: {bin(v)}")

a & b   =    8   bin: 0b1000
a | b   =   14   bin: 0b1110
a ^ b   =    6   bin: 0b110
a << 1  =   24   bin: 0b11000
a >> 2  =    3   bin: 0b11
~a      =  -13   bin: -0b1101


### Các mẹo hay dùng

| Mục đích | Biểu thức |
|---|---|
| Kiểm tra bit `k` | `(x >> k) & 1` |
| Bật / tắt / lật bit `k` | `x \| (1 << k)`, `x & ~(1 << k)`, `x ^ (1 << k)` |
| Số chẵn? | `x & 1 == 0` |
| Bỏ bit 1 thấp nhất | `x & (x - 1)` |
| Lấy bit 1 thấp nhất | `x & -x` |
| Là lũy thừa của 2? | `x > 0 and x & (x - 1) == 0` |

**Tính chất XOR** (cực kỳ hay dùng): `x ^ x = 0`, `x ^ 0 = x`, giao hoán và kết hợp → XOR cả dãy thì các số xuất hiện **chẵn** lần triệt tiêu nhau.

In [10]:
x_m = 0b10110
print("bỏ bit 1 thấp nhất:", format(x_m, "05b"), "→", format(x_m & (x_m - 1), "05b"))
print("lấy bit 1 thấp nhất:", format(x_m & -x_m, "05b"))
print("lũy thừa của 2:", [v for v in range(1, 40) if v & (v - 1) == 0])
print("đếm bit 1 của 255:", bin(255).count("1"))
xor_all = 0
for v in [4, 1, 2, 1, 2]:
    xor_all ^= v
print("XOR [4,1,2,1,2] =", xor_all, "(số xuất hiện lẻ lần)")

bỏ bit 1 thấp nhất: 10110 → 10100
lấy bit 1 thấp nhất: 00010
lũy thừa của 2: [1, 2, 4, 8, 16, 32]
đếm bit 1 của 255: 8
XOR [4,1,2,1,2] = 4 (số xuất hiện lẻ lần)


### Dùng bit làm tập hợp (bitmask)

Với `n` phần tử, một số `mask` từ `0` tới `2^n − 1` biểu diễn một **tập con**: bit `i` bằng 1 nghĩa là lấy phần tử `i`. Duyệt mọi `mask` = duyệt mọi tập con.

In [11]:
items = ["a", "b", "c"]
for mask in range(1 << len(items)):
    tap = [items[i] for i in range(len(items)) if mask >> i & 1]
    print(format(mask, "03b"), tap)

000 []
001 ['a']
010 ['b']
011 ['a', 'b']
100 ['c']
101 ['a', 'c']
110 ['b', 'c']
111 ['a', 'b', 'c']


### Số âm và giới hạn 32 bit

Ngôn ngữ như C/Java lưu `int` bằng 32 bit, số âm dạng **bù hai** (two's complement): `-x = ~x + 1`. Python có số nguyên **không giới hạn**, nên muốn mô phỏng 32 bit phải **che** bằng `& 0xFFFFFFFF` rồi đổi lại thành số có dấu.

In [12]:
def thanh_32bit(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >> 31 else x       # bit 31 là bit dấu

print("-1 ở 32 bit:", format(-1 & 0xFFFFFFFF, "032b"))
print("2^31 tràn thành:", thanh_32bit(2 ** 31), "| (2^31 - 1) + 1 =", thanh_32bit(2 ** 31 - 1 + 1))

-1 ở 32 bit: 11111111111111111111111111111111
2^31 tràn thành: -2147483648 | (2^31 - 1) + 1 = -2147483648


Lỗi hay gặp: nhầm `^` là lũy thừa (lũy thừa là `**`); quên ngoặc vì `&`, `|`, `^` có độ ưu tiên **thấp hơn** `==` — viết `(x & 1) == 0`; `~x` trong Python là `-x - 1`, không phải "đảo 32 bit".

## 2.4 Áp dụng vào bài này

**Lũy thừa nhanh**: `x^n = (x^(n/2))²` (nhân thêm `x` nếu `n` lẻ) — **chia để trị** chỉ còn `O(log n)` phép nhân. Bản lặp duyệt các **bit** của `n`: bit nào bằng 1 thì nhân `x^(2^k)` vào kết quả. Số mũ âm: tính `1 / x^(−n)`.

# 3. Nền tảng

## 3.1 Số mũ âm và số mũ 0

- `x^0 = 1`.
- `x^(-n) = 1 / x^n`. Vậy chỉ cần tính lũy thừa với số mũ **không âm**, rồi nghịch đảo nếu cần.

## 3.2 Số thực (float) không chính xác tuyệt đối

Máy tính lưu số thực ở hệ nhị phân, nên nhiều phép tính có **sai số làm tròn** rất nhỏ:

In [13]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(2.1 * 2.1 * 2.1)

import math
print(math.isclose(0.1 + 0.2, 0.3))

0.30000000000000004


False
9.261000000000001
True


Vì vậy khi kiểm tra kết quả số thực, ta dùng `math.isclose(a, b, rel_tol=...)` thay vì `==`. LeetCode cũng chấm với sai số cho phép.

# 4. Cách dễ nghĩ nhất: nhân lặp

In [14]:
from typing import List

def pow_naive(x: float, n: int) -> float:
    if n < 0:
        x, n = 1 / x, -n
    result = 1.0
    for _ in range(n):
        result *= x
    return result

print(pow_naive(2.0, 10), pow_naive(2.1, 3), pow_naive(2.0, -2))

1024.0 9.261000000000001 0.25


**Vấn đề:** `n` có thể tới `2^31 ≈ 2.1 tỷ` → 2 tỷ phép nhân. Đo thử bên dưới: với Python, chừng đó phép nhân mất **hàng chục giây**, trong khi giới hạn thời gian thường chỉ khoảng 1 giây.

In [15]:
import time
t = time.time()
pow_naive(1.0000001, 10**6)
print(f"10^6 phép nhân mất {time.time() - t:.3f}s -> 2^31 phép sẽ mất khoảng {(time.time() - t) * 2147:.0f}s")

10^6 phép nhân mất 0.030s -> 2^31 phép sẽ mất khoảng 65s


# 5. Ý tưởng: bình phương

Quan sát:

```text
x^10 = (x^5)²            -> chỉ cần tính x^5 rồi nhân với chính nó
x^5  = x · x^4 = x · (x^2)²
x^2  = (x^1)²
```

Tổng quát:

- `n` **chẵn**: `x^n = (x^(n/2))²`
- `n` **lẻ**: `x^n = x · (x^((n-1)/2))²`

Mỗi bước **chia đôi** số mũ → chỉ khoảng `log₂ n` bước. Với `n = 2^31`, chỉ ~31 bước!

In [16]:
for n in [10, 1000, 10**6, 2**31]:
    print(f"n = {n:>10}: nhân lặp cần {n:>10} phép, bình phương cần ~{n.bit_length()} bước")

n =         10: nhân lặp cần         10 phép, bình phương cần ~4 bước
n =       1000: nhân lặp cần       1000 phép, bình phương cần ~10 bước
n =    1000000: nhân lặp cần    1000000 phép, bình phương cần ~20 bước
n = 2147483648: nhân lặp cần 2147483648 phép, bình phương cần ~32 bước


# 6. Cách 1: đệ quy

```python
def fast_pow(x, n):          # n >= 0
    if n == 0:
        return 1.0
    half = fast_pow(x, n // 2)
    if n % 2 == 0:
        return half * half
    return half * half * x
```

**Quan trọng:** gọi `fast_pow(x, n // 2)` **một lần** và lưu vào `half`. Nếu viết `fast_pow(x, n//2) * fast_pow(x, n//2)`, số lời gọi lại thành `O(n)`!

In [17]:
def fast_pow_recursive(x: float, n: int) -> float:
    if n == 0:
        return 1.0
    half = fast_pow_recursive(x, n // 2)
    if n % 2 == 0:
        return half * half
    return half * half * x


def my_pow_recursive(x: float, n: int) -> float:
    if n < 0:
        return 1.0 / fast_pow_recursive(x, -n)
    return fast_pow_recursive(x, n)

print(my_pow_recursive(2.0, 10), my_pow_recursive(2.1, 3), my_pow_recursive(2.0, -2))

1024.0 9.261000000000001 0.25


# 7. Mô phỏng đệ quy: `2^10`

```text
pow(2, 10): chẵn -> pow(2, 5)²
  pow(2, 5): lẻ  -> pow(2, 2)² · 2
    pow(2, 2): chẵn -> pow(2, 1)²
      pow(2, 1): lẻ -> pow(2, 0)² · 2 = 1 · 2 = 2
    pow(2, 2) = 2² = 4
  pow(2, 5) = 4² · 2 = 32
pow(2, 10) = 32² = 1024
```

In [18]:
def trace_fast_pow(x: float, n: int, depth: int = 0) -> float:
    indent = "  " * depth
    if n == 0:
        print(f"{indent}pow({x}, 0) = 1")
        return 1.0
    kind = "chẵn" if n % 2 == 0 else "lẻ"
    print(f"{indent}pow({x}, {n}): {kind} -> cần pow({x}, {n // 2})")
    half = trace_fast_pow(x, n // 2, depth + 1)
    result = half * half if n % 2 == 0 else half * half * x
    formula = f"{half}²" if n % 2 == 0 else f"{half}² · {x}"
    print(f"{indent}pow({x}, {n}) = {formula} = {result}")
    return result

_ = trace_fast_pow(2, 10)

pow(2, 10): chẵn -> cần pow(2, 5)
  pow(2, 5): lẻ -> cần pow(2, 2)
    pow(2, 2): chẵn -> cần pow(2, 1)
      pow(2, 1): lẻ -> cần pow(2, 0)
        pow(2, 0) = 1
      pow(2, 1) = 1.0² · 2 = 2.0
    pow(2, 2) = 2.0² = 4.0
  pow(2, 5) = 4.0² · 2 = 32.0
pow(2, 10) = 32.0² = 1024.0


# 8. Cách 2: vòng lặp theo các bit của `n`

Viết `n` ở hệ nhị phân. Ví dụ `n = 10 = 0b1010 = 8 + 2`:

```text
x^10 = x^8 · x^2
```

Ta lần lượt có `x^1, x^2, x^4, x^8, …` bằng cách **bình phương liên tiếp**, và nhân vào kết quả những lũy thừa ứng với **bit 1** của `n`.

```python
result = 1
while n > 0:
    if n & 1:          # bit thấp nhất là 1
        result *= x
    x *= x             # x, x^2, x^4, x^8, ...
    n >>= 1            # bỏ bit thấp nhất
```

`n & 1` lấy bit cuối (giống `n % 2`); `n >>= 1` dịch phải một bit (giống `n //= 2`).

In [19]:
def my_pow(x: float, n: int) -> float:
    if n < 0:
        x, n = 1.0 / x, -n
    result = 1.0
    while n > 0:
        if n & 1:
            result *= x
        x *= x
        n >>= 1
    return result

print(my_pow(2.0, 10), my_pow(2.1, 3), my_pow(2.0, -2))

1024.0 9.261000000000001 0.25


In [20]:
def trace_my_pow(x: float, n: int) -> float:
    print(f"n = {n} = {bin(n)}")
    result = 1.0
    power = x
    bit = 0
    while n > 0:
        take = n & 1
        if take:
            result *= power
        print(f"bit {bit}: {take} | lũy thừa hiện tại x^{2**bit} = {power:<10} | {'nhân vào' if take else 'bỏ qua':9} | result = {result}")
        power *= power
        n >>= 1
        bit += 1
    return result

_ = trace_my_pow(2, 10)

n = 10 = 0b1010
bit 0: 0 | lũy thừa hiện tại x^1 = 2          | bỏ qua    | result = 1.0
bit 1: 1 | lũy thừa hiện tại x^2 = 4          | nhân vào  | result = 4.0
bit 2: 0 | lũy thừa hiện tại x^4 = 16         | bỏ qua    | result = 4.0
bit 3: 1 | lũy thừa hiện tại x^8 = 256        | nhân vào  | result = 1024.0


# 9. Phiên bản đúng format LeetCode

In [21]:
class Solution:
    def myPow(self, x: float, n: int) -> float:
        if n < 0:
            x, n = 1.0 / x, -n
        result = 1.0
        while n:
            if n & 1:
                result *= x
            x *= x
            n >>= 1
        return result

In [22]:
def check(name, actual, expected):
    assert math.isclose(actual, expected, rel_tol=1e-9, abs_tol=1e-12), (name, actual, expected)
    print(f"{name:<28} PASS  {actual:.5f}")


sol = Solution()
check("Ví dụ 1: 2^10", sol.myPow(2.0, 10), 1024.0)
check("Ví dụ 2: 2.1^3", sol.myPow(2.1, 3), 9.261)
check("Ví dụ 3: 2^-2", sol.myPow(2.0, -2), 0.25)
check("Mũ 0", sol.myPow(5.5, 0), 1.0)
check("Cơ số 1, mũ cực lớn", sol.myPow(1.0, 2**31 - 1), 1.0)
check("Cơ số -1, mũ lẻ", sol.myPow(-1.0, 2**31 - 1), -1.0)
check("Mũ âm nhỏ nhất", sol.myPow(1.0, -2**31), 1.0)
check("Cơ số âm, mũ chẵn", sol.myPow(-2.0, 4), 16.0)
check("Cơ số 0, mũ dương", sol.myPow(0.0, 5), 0.0)

Ví dụ 1: 2^10                PASS  1024.00000
Ví dụ 2: 2.1^3               PASS  9.26100
Ví dụ 3: 2^-2                PASS  0.25000
Mũ 0                         PASS  1.00000
Cơ số 1, mũ cực lớn          PASS  1.00000
Cơ số -1, mũ lẻ              PASS  -1.00000
Mũ âm nhỏ nhất               PASS  1.00000
Cơ số âm, mũ chẵn            PASS  16.00000
Cơ số 0, mũ dương            PASS  0.00000


In [23]:
import random

random.seed(50)
for _ in range(5000):
    x = random.uniform(-3, 3)
    n = random.randint(-20, 20)
    if x == 0 and n <= 0:
        continue
    expected = x ** n
    assert math.isclose(sol.myPow(x, n), expected, rel_tol=1e-9, abs_tol=1e-12), (x, n)
    assert math.isclose(my_pow_recursive(x, n), expected, rel_tol=1e-9, abs_tol=1e-12), (x, n)
print("5000 test ngẫu nhiên khớp toán tử ** (với sai số cho phép)")

t = time.time()
sol.myPow(1.0000001, 2**31 - 1)
print(f"n = 2^31 - 1 chạy trong {time.time() - t:.6f}s")

5000 test ngẫu nhiên khớp toán tử ** (với sai số cho phép)
n = 2^31 - 1 chạy trong 0.000101s


# 10. Vì sao đúng? Độ phức tạp?

**Bản vòng lặp — bất biến:** gọi `x₀`, `n₀` là giá trị ban đầu. Ở đầu mỗi vòng:

```text
result · x^n = x₀^n₀
```

- Ban đầu `result = 1`, đúng.
- Nếu bit cuối của `n` là 1: `x^n = x · x^(n-1)` → chuyển một thừa số `x` vào `result`.
- Rồi `x^n` (với `n` giờ chẵn) `= (x²)^(n/2)` → `x *= x`, `n >>= 1`.

Khi `n = 0`: `result · 1 = x₀^n₀`. Vậy `result` là đáp án.

**Độ phức tạp:**

```text
Time:  O(log |n|)     — mỗi vòng bỏ một bit; tối đa 32 vòng
Space: O(1) (vòng lặp) / O(log |n|) (đệ quy, cho call stack)
```

# 11. Những lỗi dễ mắc

## Lỗi 1: Gọi đệ quy hai lần

In [24]:
calls = 0

def wrong_double_call(x: float, n: int) -> float:
    global calls
    calls += 1
    if n == 0:
        return 1.0
    if n % 2 == 0:
        return wrong_double_call(x, n // 2) * wrong_double_call(x, n // 2)   # SAI: gọi 2 lần
    return wrong_double_call(x, n // 2) * wrong_double_call(x, n // 2) * x

calls = 0
wrong_double_call(1.0, 1000)
print("Gọi hai lần: số lời gọi =", calls)

calls_ok = 0
def counted(x, n):
    global calls_ok
    calls_ok += 1
    if n == 0:
        return 1.0
    half = counted(x, n // 2)
    return half * half * (x if n % 2 else 1)
counted(1.0, 1000)
print("Gọi một lần: số lời gọi =", calls_ok)

Gọi hai lần: số lời gọi = 2047
Gọi một lần: số lời gọi = 11


Gọi hai lần biến cây đệ quy thành cây nhị phân đầy đủ → `O(n)` lời gọi, mất hết lợi ích.

## Lỗi 2: Đổi dấu `n` trong ngôn ngữ 32-bit

`n = -2^31` → `-n = 2^31` **tràn** `int` 32-bit (Java/C++). Phải chuyển sang `long` trước khi đổi dấu. Python không gặp vấn đề này.

## Lỗi 3: So sánh float bằng `==`

`sol.myPow(2.1, 3) == 9.261` có thể là `False` do sai số làm tròn:

In [25]:
print(sol.myPow(2.1, 3), sol.myPow(2.1, 3) == 9.261, math.isclose(sol.myPow(2.1, 3), 9.261))

9.261000000000001

 False True


## Lỗi 4: Quên trường hợp `n = 0`

Vòng `while n:` không chạy, trả `1.0` — đúng tự nhiên. Nhưng bản đệ quy thiếu base case `n == 0` sẽ đệ quy mãi.

# 12. Mẹo nhận diện pattern

**Lũy thừa nhanh** khi cần áp dụng một phép **kết hợp được** (associative) rất nhiều lần:

| Bài toán | "Nhân" là gì |
|---|---|
| `x^n` (#50) | nhân số |
| `a^b mod m` (#372 Super Pow) | nhân rồi lấy dư |
| Fibonacci thứ `n` trong `O(log n)` | nhân ma trận `2 × 2` |
| Áp dụng một phép biến đổi `n` lần | hợp các phép biến đổi |

Họ hàng gần: **#29 Divide Two Integers** (nhân đôi số chia), **#69 Sqrt(x)** (binary search).

## Template

```python
result = identity
while n:
    if n & 1: result = combine(result, base)
    base = combine(base, base)
    n >>= 1
```

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Trace bản vòng lặp cho `x = 3`, `n = 13` (`13 = 0b1101`). Bao nhiêu phép nhân?

## Bài 2 — Lũy thừa số nguyên modulo

Tính `(a^b) mod m` với `a, b, m` nguyên dương, `b` tới `10^18`. Không dùng `pow` có sẵn.

```python
pow_mod(2, 10, 1000)        -> 24
pow_mod(3, 10**18, 10**9+7) -> pow(3, 10**18, 10**9+7)
```

In [26]:
def exercise_2_pow_mod(a: int, b: int, m: int) -> int:
    # TODO: như my_pow, nhưng sau mỗi phép nhân lấy % m
    pass


# assert exercise_2_pow_mod(2, 10, 1000) == 24
# assert exercise_2_pow_mod(3, 10**18, 10**9 + 7) == pow(3, 10**18, 10**9 + 7)
# assert exercise_2_pow_mod(5, 0, 7) == 1

## Bài 3 — Điền chỗ trống

In [27]:
def exercise_3_my_pow(x: float, n: int) -> float:
    # TODO 1: xử lý n âm
    result = 1.0
    # TODO 2: vòng lặp theo bit: nhân vào result nếu bit = 1, bình phương x, dịch n
    return result


# assert math.isclose(exercise_3_my_pow(2.0, 10), 1024.0)
# assert math.isclose(exercise_3_my_pow(2.0, -2), 0.25)
# assert exercise_3_my_pow(7.0, 0) == 1.0

## Bài 4 — Biến thể: Fibonacci bằng lũy thừa ma trận

```text
[F(n+1)  F(n)  ]   =   [1 1]^n
[F(n)    F(n-1)]       [1 0]
```

Tính `F(n) mod (10^9 + 7)` trong `O(log n)`.

```python
fib_mod(10)    -> 55
fib_mod(100)   -> 687995182
```

In [28]:
def exercise_4_fib_mod(n: int) -> int:
    # TODO: viết hàm nhân ma trận 2x2 mod M, rồi lũy thừa nhanh ma trận [[1,1],[1,0]]
    pass


# assert exercise_4_fib_mod(0) == 0
# assert exercise_4_fib_mod(10) == 55
# assert exercise_4_fib_mod(100) == 687995182

## Bài 5 — Mini challenge: Sqrt(x) (LeetCode #69)

Tính phần nguyên của `√x` (`x` nguyên không âm) **không dùng** `sqrt` hay `**`. Dùng binary search tìm số `r` lớn nhất có `r * r <= x`.

```python
my_sqrt(8)  -> 2
my_sqrt(16) -> 4
my_sqrt(0)  -> 0
```

In [29]:
def exercise_5_my_sqrt(x: int) -> int:
    # TODO
    pass


# assert exercise_5_my_sqrt(8) == 2
# assert exercise_5_my_sqrt(16) == 4
# assert exercise_5_my_sqrt(0) == 0
# assert exercise_5_my_sqrt(2**31 - 1) == 46340

# 14. Gợi ý

**Bài 1**
- Mức 1: các bit 1 ở vị trí 0, 2, 3 → `3^1 · 3^4 · 3^8`.

**Bài 2**
- Mức 1: `result = result * a % m`, `a = a * a % m`.

**Bài 3**
- Mức 1: so với `Solution` mục 9.

**Bài 4**
- Mức 1: `mat_mul(A, B)` trả về ma trận `2 × 2` với từng phần tử `% M`.
- Mức 2: kết quả ban đầu là ma trận đơn vị `[[1, 0], [0, 1]]`.
- Mức 3: `F(n)` là phần tử `[0][1]` của `[[1,1],[1,0]]^n`.

**Bài 5**
- Mức 1: tìm trong `[0, x]`; điều kiện `mid * mid <= x` là đơn điệu (True…True False…False).
- Mức 2: dùng `lo, hi = 0, x`; `mid = (lo + hi + 1) // 2` để tránh lặp vô hạn khi `lo = mid`.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [30]:
_ = trace_my_pow(3, 13)
print("Kiểm tra:", 3 ** 13)

n = 13 = 0b1101
bit 0: 1 | lũy thừa hiện tại x^1 = 3          | nhân vào  | result = 3.0
bit 1: 0 | lũy thừa hiện tại x^2 = 9          | bỏ qua    | result = 3.0
bit 2: 1 | lũy thừa hiện tại x^4 = 81         | nhân vào  | result = 243.0
bit 3: 1 | lũy thừa hiện tại x^8 = 6561       | nhân vào  | result = 1594323.0
Kiểm tra: 1594323


Có 3 lần nhân vào `result` (bit 0, 2, 3) và 4 lần bình phương → tổng 7 phép nhân (so với 12 nếu nhân lặp). Với `n` lớn, khoảng cách còn lớn hơn rất nhiều.

## Đáp án Bài 2

In [31]:
def answer_2_pow_mod(a: int, b: int, m: int) -> int:
    result = 1 % m
    a %= m
    while b:
        if b & 1:
            result = result * a % m
        a = a * a % m
        b >>= 1
    return result

assert answer_2_pow_mod(2, 10, 1000) == 24
assert answer_2_pow_mod(3, 10**18, 10**9 + 7) == pow(3, 10**18, 10**9 + 7)
assert answer_2_pow_mod(5, 0, 7) == 1
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [32]:
def answer_3_my_pow(x: float, n: int) -> float:
    if n < 0:
        x, n = 1.0 / x, -n
    result = 1.0
    while n:
        if n & 1:
            result *= x
        x *= x
        n >>= 1
    return result

assert math.isclose(answer_3_my_pow(2.0, 10), 1024.0)
assert math.isclose(answer_3_my_pow(2.0, -2), 0.25)
assert answer_3_my_pow(7.0, 0) == 1.0
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [33]:
MOD = 10**9 + 7

def mat_mul(A, B):
    return [
        [(A[0][0] * B[0][0] + A[0][1] * B[1][0]) % MOD, (A[0][0] * B[0][1] + A[0][1] * B[1][1]) % MOD],
        [(A[1][0] * B[0][0] + A[1][1] * B[1][0]) % MOD, (A[1][0] * B[0][1] + A[1][1] * B[1][1]) % MOD],
    ]

def answer_4_fib_mod(n: int) -> int:
    result = [[1, 0], [0, 1]]
    base = [[1, 1], [1, 0]]
    while n:
        if n & 1:
            result = mat_mul(result, base)
        base = mat_mul(base, base)
        n >>= 1
    return result[0][1]

assert answer_4_fib_mod(0) == 0
assert answer_4_fib_mod(10) == 55
assert answer_4_fib_mod(100) == 687995182
a, b = 0, 1
for k in range(200):
    assert answer_4_fib_mod(k) == a % MOD
    a, b = b, a + b
print("Bài 4 PASS (khớp Fibonacci tính tuần tự với n = 0..199), F(10^18) mod M =", answer_4_fib_mod(10**18))

Bài 4 PASS (khớp Fibonacci tính tuần tự với n = 0..199), F(10^18) mod M = 209783453


## Đáp án Bài 5

In [34]:
def answer_5_my_sqrt(x: int) -> int:
    lo, hi = 0, x
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if mid * mid <= x:
            lo = mid
        else:
            hi = mid - 1
    return lo

assert answer_5_my_sqrt(8) == 2
assert answer_5_my_sqrt(16) == 4
assert answer_5_my_sqrt(0) == 0
assert answer_5_my_sqrt(2**31 - 1) == 46340
assert all(answer_5_my_sqrt(v) == math.isqrt(v) for v in range(5000))
print("Bài 5 PASS (khớp math.isqrt với 0..4999)")

Bài 5 PASS (khớp math.isqrt với 0..4999)


# 16. Tổng kết

> `x^n = (x^(n/2))²` (nhân thêm `x` nếu `n` lẻ). Mỗi bước chia đôi số mũ → `O(log n)`. Bản vòng lặp: bình phương liên tiếp `x, x², x⁴, …`, nhân vào kết quả những lũy thừa ứng với **bit 1** của `n`. Số mũ âm → nghịch đảo `x`.

---

## Checklist trước khi nộp

- [ ] Xử lý `n < 0` bằng `x = 1 / x`, `n = -n`?
- [ ] Đệ quy chỉ gọi **một lần** cho `n // 2`?
- [ ] `n = 0` trả `1.0`?
- [ ] (Ngôn ngữ 32-bit) đổi `n` sang `long` trước khi đổi dấu?
- [ ] Kiểm tra kết quả float bằng `math.isclose`?